![CREWS UFFFS Nowcast](logo.png)

# CREWS UFFFS Workshop — NOAA Enterprise Rain Rate (Himawari blend)

In this tutorial you will build an end-to-end satellite precipitation
nowcasting workflow for **Phnom Penh** (Cambodia) and **Vientiane**
(Lao PDR), using the **NOAA Enterprise Rain Rate (ERR)** product on the
Himawari sector. You will:

1. Understand the ERR product and why it is well-suited to urban flash-flood
   forecasting.
2. Download a window of 10-minute ERR rainfall fields directly from the
   public NOAA Open Data Dissemination (NODD) S3 bucket — no AWS account
   needed.
3. Combine the files into a single, area-of-interest (AOI) NetCDF and
   inspect the rainfall observations spatially and as time series at the
   two cities.
4. Run a **pySTEPS ensemble nowcast** of the most recent observations.
5. Visualise the nowcast as a spatial field and as a combined
   observations + nowcast time series at each city.

> **About this notebook.** All the technical Python functions used by the
> notebook live in the file `workshop_helpers.py`. You do not need to read
> that file to follow the workshop; the notebook imports the helpers and
> calls them where needed. All the spatial maps below are **interactive
> Bokeh plots**: you can pan, zoom, and hover over pixels to read longitude,
> latitude and rainfall values.

## Step 1: What is the NOAA Enterprise Rain Rate (ERR) product?

The **NOAA Enterprise Rain Rate (ERR)**, sometimes still called by its
algorithm name **SCaMPR** (*Self-Calibrating Multivariate Precipitation
Retrieval*), is an operational geostationary-satellite rainfall product
produced by NOAA/NESDIS. The product is the successor to the legacy NOAA
Global Hydro-Estimator (GHE).

| Property              | Value                                                                                       |
| --------------------- | ------------------------------------------------------------------------------------------- |
| Spatial resolution    | **2 km** at nadir                                                                           |
| Temporal cadence      | **every 10 minutes**                                                                        |
| Variable              | Instantaneous rainfall rate, in mm h⁻¹                                                      |
| Coverage              | Full disk; global mosaic 70°N – 60°S, split into tiles (GLB-2 … GLB-5)                       |
| Source over Asia      | **Himawari-9** (Himawari-8 as backup) at 140.7°E                                            |
| Latency               | Typically ~5 minutes after scan end                                                         |
| Algorithm             | IR predictors calibrated against passive-microwave rain rates (CPC MWCOMB), with GFS-RH adjustment for sub-cloud evaporation |
| Access                | Public AWS S3 bucket `noaa-enterprise-rainrate-pds`, anonymous read                          |

### Why this product for UFFFS?

- The combination of **2 km resolution** and **10-minute cadence** means
  the product can resolve individual convective cells over urban catchments
  such as Phnom Penh and Vientiane.
- The **low latency** (3–6 minutes after scan-end in the files we observe)
  preserves the available nowcast lead time.
- The product is already on a **regular lat/lon grid** — no satellite-projection
  reprojection is required before feeding it into pySTEPS.
- Access is **completely open**: no AWS account, no registration, no FTP
  credentials. The same workflow can be deployed by national hydromet
  services with no commercial dependencies.

### Tile structure

The ERR blend is delivered as four global tiles, one per source geostationary
satellite. For Cambodia and Lao PDR the relevant tile is **GLB-5**, which is
generated from Himawari and covers approximately 80°E – 180° longitude.

### Known limitations

- The product is **IR-anchored** — like all IR-based rainfall retrievals,
  it can underestimate warm-rain (low-cloud-top) precipitation and
  false-alarm on cold non-raining cirrus.
- The **microwave calibration is global**, not tropical-specific. For
  operational use in Cambodia / Lao PDR, gauge bias-correction against
  the DOM / DMH AWS networks is recommended.
- The product is **instantaneous rain rate**; users wanting hourly
  accumulations should integrate the 10-min snapshots.

### What we will do in this notebook

For workshop purposes we will download a recent 4-hour window of ERR
data, combine it into one NetCDF on a regional bounding box, visualise
it, and run a pySTEPS ensemble nowcast with six 10-minute lead times
(i.e. a 1-hour forecast horizon).

## Step 2: Import libraries and helper functions

We use:

- `boto3` for anonymous S3 access (called from inside the helpers);
- `xarray`, `pandas`, `numpy` for the rainfall data;
- `bokeh` (initialised below) for the interactive plots;
- `pysteps` for optical-flow nowcasting (called from inside the helpers);
- `workshop_helpers` for all the project-specific code.

In [1]:
import warnings
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime, timedelta, timezone

import workshop_helpers as wh

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Initialise Bokeh for inline rendering inside this notebook.
wh.init_bokeh_notebook()

print("Libraries and helpers imported.")

Libraries and helpers imported.


## Step 3: Configure the event window, AOI, and nowcast settings

Most workshop discussions will involve changing one or more of these
settings. Read them once carefully, then move on.

### Time window

We take a **4-hour window** of ERR observations ending one hour before
the current UTC time. The trailing one-hour buffer gives the operational
ERR pipeline plenty of time to finish writing the latest files. For a
case-study replay, set `EVENT_START` and `EVENT_END` to explicit ISO
timestamps.

### Area of interest (AOI)

The bounding box covers **Cambodia, Lao PDR and parts of Thailand and
Vietnam**, large enough to see synoptic rainfall systems moving towards
both cities. The two city zoom locations are city-centre reference points.

### Nowcast settings

- `n_input_files`: how many of the most recent observed fields are used
  to estimate motion. With 10-minute data, **6 input files = the last
  60 minutes**.
- `n_lead_times`: how many forecast timesteps to produce. With 10-minute
  data, **6 lead times = a 60-minute forecast horizon**.
- `n_ensemble_members`: the number of stochastic STEPS members. More
  members give more robust uncertainty information at the cost of
  run time.
- `km_per_pixel`: ERR is 2 km at nadir; we use 2 km here so STEPS'
  stochastic perturbations are scaled correctly.

In [2]:
# ---------------------------------------------------------------------
# Event window (UTC). By default: the 4-hour window ending one hour ago.
# For a fixed replay, set EVENT_START and EVENT_END to ISO timestamps.
# ---------------------------------------------------------------------
_now = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
EVENT_END   = (_now - timedelta(hours=1)).strftime("%Y-%m-%dT%H:%M:%SZ")
EVENT_START = (_now - timedelta(hours=7)).strftime("%Y-%m-%dT%H:%M:%SZ")

# Example fixed-event replay (uncomment to use):
# EVENT_START = "2026-05-10T00:00:00Z"
# EVENT_END   = "2026-05-10T06:00:00Z"


# EVENT_START = "2025-10-07T13:00:00Z"
# EVENT_END   = "2025-10-07T19:00:00Z"

print(f"Event start (UTC): {EVENT_START}")
print(f"Event end   (UTC): {EVENT_END}")

Event start (UTC): 2026-06-10T04:00:00Z
Event end   (UTC): 2026-06-10T10:00:00Z


In [3]:
settings = {
    # -----------------------------------------------------------------
    # Workshop metadata
    # -----------------------------------------------------------------
    "project_name": "CREWS_UFFFS_training",
    "country_or_region": "Southeast Asia (Cambodia + Lao PDR)",
    "event_name": "Recent 4-hour NOAA ERR window",
    "source_product": "NOAA Enterprise Rain Rate (ERR) - Himawari blend",

    # -----------------------------------------------------------------
    # ERR download
    # -----------------------------------------------------------------
    "err_tile": 5,                       # GLB-5 = Himawari sector
    "native_timestep_minutes": 10,       # ERR cadence

    # -----------------------------------------------------------------
    # Spatial domain and city zoom locations
    # -----------------------------------------------------------------
    # bbox = [min_lon, min_lat, max_lon, max_lat]
    "bbox": [98.0, 8.0, 110.0, 22.0],

    "city_zoom_enabled": True,
    "city_zoom_buffer_degrees": 1.0,
    "city_zoom_locations": [
        {"name": "Phnom Penh", "country": "Cambodia",
         "lat": 11.5564, "lon": 104.9282,
         "notes": "City-centre reference point."},
        {"name": "Vientiane",  "country": "Lao PDR",
         "lat": 17.9757, "lon": 102.6331,
         "notes": "City-centre reference point."},
    ],

    # -----------------------------------------------------------------
    # Bokeh plotting options
    # -----------------------------------------------------------------
    "plot_zero_as_nan":     True,
    "plot_use_osm_basemap": True,
    "plot_rainfall_alpha":  0.7,
    "plot_vmax_percentile": 99.0,
    "plot_intensity_vmax":  None,

    # -----------------------------------------------------------------
    # Event window (set above)
    # -----------------------------------------------------------------
    "start_time": EVENT_START,
    "end_time":   EVENT_END,

    # -----------------------------------------------------------------
    # Nowcast configuration (ERR is 10-min cadence -> each lead is 10 min)
    # -----------------------------------------------------------------
    "nowcast_reference_time": None,  # None = use latest available field
    "n_input_files": 35,              # 6 x 10 min = last 60 min of obs
    "n_lead_times": 36,               # 6 x 10 min = 60-min forecast horizon
    "timestep_minutes": 10,

    # Sub-threshold rainfall is treated as zero for motion estimation.
    "rain_threshold_mm_h": 0.1,

    "use_steps_if_available": True,
    "n_ensemble_members": 8,

    # ERR grid spacing in km (used by STEPS stochastic perturbations).
    "km_per_pixel": 2.0,

    "random_seed": 42,
}
settings

{'project_name': 'CREWS_UFFFS_training',
 'country_or_region': 'Southeast Asia (Cambodia + Lao PDR)',
 'event_name': 'Recent 4-hour NOAA ERR window',
 'source_product': 'NOAA Enterprise Rain Rate (ERR) - Himawari blend',
 'err_tile': 5,
 'native_timestep_minutes': 10,
 'bbox': [98.0, 8.0, 110.0, 22.0],
 'city_zoom_enabled': True,
 'city_zoom_buffer_degrees': 1.0,
 'city_zoom_locations': [{'name': 'Phnom Penh',
   'country': 'Cambodia',
   'lat': 11.5564,
   'lon': 104.9282,
   'notes': 'City-centre reference point.'},
  {'name': 'Vientiane',
   'country': 'Lao PDR',
   'lat': 17.9757,
   'lon': 102.6331,
   'notes': 'City-centre reference point.'}],
 'plot_zero_as_nan': True,
 'plot_use_osm_basemap': True,
 'plot_rainfall_alpha': 0.7,
 'plot_vmax_percentile': 99.0,
 'plot_intensity_vmax': None,
 'start_time': '2026-06-10T04:00:00Z',
 'end_time': '2026-06-10T10:00:00Z',
 'nowcast_reference_time': None,
 'n_input_files': 35,
 'n_lead_times': 36,
 'timestep_minutes': 10,
 'rain_threshold_

## Step 4: Download ERR files from the public S3 bucket

The cell below lists every ERR file with a scan-start time inside the
event window, and downloads any that are not already present on disk.

This step talks directly to the public bucket
`s3://noaa-enterprise-rainrate-pds/` using anonymous (unsigned) S3
requests — no AWS credentials are needed.

Typical timing: a 4-hour window contains roughly 24 files at ~3 MB each
(≈ 70 MB total) and downloads in a couple of minutes on a workshop
laptop.

In [4]:
folders = wh.setup_folders("./err_data")
folders

{'data': WindowsPath('err_data'),
 'raw': WindowsPath('err_data/raw_netcdf'),
 'processed': WindowsPath('err_data/processed_netcdf'),
 'nowcast': WindowsPath('err_data/nowcast'),
 'figures': WindowsPath('err_data/figures')}

In [5]:
file_entries = wh.list_err_files_for_window(
    start=settings["start_time"],
    end=settings["end_time"],
    tile=settings["err_tile"],
)
print(f"{len(file_entries)} ERR files match the event window for tile {settings['err_tile']}.")
if file_entries:
    print()
    print("First file:", file_entries[0]["start"], file_entries[0]["key"])
    print("Last  file:", file_entries[-1]["start"], file_entries[-1]["key"])

36 ERR files match the event window for tile 5.

First file: 202606100400000 BLEND/RainRate-Blend-INST/2026/06/10/04/RRQPE-INST-GLB-5_v1r1_blend_s202606100400000_e202606100409599_c202606100422334.nc
Last  file: 202606100950000 BLEND/RainRate-Blend-INST/2026/06/10/09/RRQPE-INST-GLB-5_v1r1_blend_s202606100950000_e202606100959599_c202606101005571.nc


In [6]:
nc_files = wh.download_err_files(file_entries, folders["raw"])

Downloaded or found 36 ERR files in err_data\raw_netcdf.


## Step 5: Combine the files into one workshop NetCDF

Each ERR file is a global tile. We:

1. Open each file with `xarray`;
2. Subset to the AOI bounding box;
3. Stack the time dimension;
4. Save the result as a single compressed NetCDF that the rest of the
   notebook (and Tutorial 3, if you build one) can re-open.

In [7]:
processed_file, err_ds = wh.combine_err_netcdfs(
    nc_files, folders["processed"], settings,
)
err_ds

Reading ERR NetCDFs:   0%|          | 0/36 [00:00<?, ?it/s]

Saved combined ERR dataset: err_data\processed_netcdf\NOAA_ERR_blend_GLB5_20260610T0400_20260610T0950.nc


<xarray.Dataset> Size: 60MB
Dimensions:             (lat: 700, lon: 600, time: 36)
Coordinates:
  * lat                 (lat) float64 6kB 8.0 8.02 8.04 ... 21.94 21.96 21.98
  * lon                 (lon) float64 5kB 98.02 98.04 98.06 ... 110.0 110.0
  * time                (time) datetime64[ns] 288B 2026-06-10T04:00:00 ... 20...
Data variables:
    precipitation_rate  (time, lat, lon) float32 60MB 0.0 0.0 0.0 ... 0.0 0.0
Attributes:
    project_name:       CREWS_UFFFS_training
    country_or_region:  Southeast Asia (Cambodia + Lao PDR)
    event_name:         Recent 4-hour NOAA ERR window
    source_product:     NOAA Enterprise Rain Rate (ERR / SCaMPR) -- Himawari ...
    source_bucket:      noaa-enterprise-rainrate-pds
    bbox:               [98.0, 8.0, 110.0, 22.0]
    tile:               5
    timestep_minutes:   10

### Quality-control summary

The table below reports basic statistics on the ERR dataset. Useful sanity
checks:

- `Number of timesteps` should be the expected count for your window
  (≈ 6 files per hour for ERR).
- `Maximum rainfall rate` should be physically plausible; values above
  ~200 mm h⁻¹ are very rare for IR-anchored retrievals.
- `Wet-pixel fraction` is the fraction of all (time × space) pixels above
  the rain threshold; in the dry season this can be very low, in the
  monsoon it climbs to a few %.

In [8]:
qc = wh.summarize_rainfall_quality(err_ds, threshold_mm_h=0.5)
qc

,diagnostic,value,unit
0,Number of timesteps,3.600000e+01,10-min steps
1,Minimum rainfall rate,0.000000e+00,mm h-1
2,Mean rainfall rate,3.043977e-01,mm h-1
3,Maximum rainfall rate,4.340000e+01,mm h-1
4,Wet pixels above threshold,1.169017e+06,pixels >= 0.5 mm h-1
5,Wet-pixel fraction,7.731594e-02,fraction of all time-space pixels
6,Missing pixels,0.000000e+00,pixels
7,Missing-pixel fraction,0.000000e+00,fraction of all time-space pixels


## Step 6: Visualise the observations spatially

We plot:

1. The **latest** ERR rainfall field over the full AOI (Cambodia, Lao PDR
   and neighbours) with city markers.
2. The same latest field **zoomed in** on each configured city.
3. The **maximum** rainfall rate over the entire event window — a useful
   single-panel summary that highlights where the heaviest rainfall
   occurred during the workshop window.

A shared colour scale (`obs_vmax`) is computed from a high percentile of
the data so the different panels can be compared directly.

In [9]:
rain = err_ds["precipitation_rate"]

latest_field = rain.isel(time=-1)
max_field    = rain.max("time")

obs_vmax = wh.compute_vmax(
    [latest_field, max_field],
    percentile=settings["plot_vmax_percentile"],
    override=settings.get("plot_intensity_vmax"),
)
print(f"Shared observation colour-bar max: {obs_vmax:.2f} mm/h"
      if obs_vmax else "All pixels are below threshold for this window.")

Shared observation colour-bar max: 28.10 mm/h


In [10]:
wh.plot_field_bokeh(
    latest_field,
    settings,
    title=(
        f"NOAA ERR — latest observation | "
        f"{pd.to_datetime(latest_field.time.values)} UTC"
    ),
    vmax=obs_vmax,
    cbar_label="mm h-1",
)

figure(id='p1009', ...)

### Latest observation zoomed around each city

In [11]:
wh.plot_city_zooms_bokeh(
    latest_field,
    settings,
    title_prefix=(
        f"NOAA ERR latest | {pd.to_datetime(latest_field.time.values)} UTC"
    ),
    vmax=obs_vmax,
    cbar_label="mm h-1",
)

[figure(id='p1093', ...), figure(id='p1165', ...)]

### Maximum observed rainfall rate over the event window

In [12]:
wh.plot_field_bokeh(
    max_field,
    settings,
    title="NOAA ERR — maximum observed rainfall rate over the event window",
    vmax=obs_vmax,
    cbar_label="mm h-1",
)

figure(id='p1237', ...)

In [13]:
wh.plot_city_zooms_bokeh(
    max_field,
    settings,
    title_prefix="NOAA ERR maximum over event window",
    vmax=obs_vmax,
    cbar_label="mm h-1",
)

[figure(id='p1321', ...), figure(id='p1393', ...)]

## Step 7: Time series of observed rainfall at each city

The plot below extracts rainfall from the **nearest ERR grid cell** to
each city centre and draws a time series. Click a legend entry to hide
or show that city.

Two complementary views:

1. **Rainfall rate** (instantaneous, mm h⁻¹) — the raw 10-minute time
   series. Spikes correspond to convective cells passing over the cell.
2. **Cumulative rainfall** (mm) — running event accumulation. Useful for
   flood-risk discussions: the slope of the cumulative line is the
   intensity, but the height is what matters for soil saturation and
   urban-drainage stress.

In [14]:
wh.plot_city_timeseries_bokeh(
    err_ds, settings,
    title="NOAA ERR rainfall rate at city locations (10-min cadence)",
)

figure(id='p1464', ...)

In [15]:
wh.plot_city_accumulation_timeseries_bokeh(
    err_ds, settings,
    title="Cumulative NOAA ERR rainfall at city locations",
)

figure(id='p1555', ...)

## Step 8: Select the input fields for the nowcast

pySTEPS estimates motion from a short sequence of recent rainfall fields.
With `n_input_files = 6` and 10-minute data, the nowcast looks at the
last 60 minutes.

If `settings["nowcast_reference_time"]` is `None`, the latest available
timestep is used as the last observation. To run a hindcast at an earlier
moment, set this to an ISO timestamp present in the dataset.

In [16]:
input_ds = wh.select_nowcast_input(
    err_ds,
    settings["n_input_files"],
    reference_time=settings["nowcast_reference_time"],
)

print("Selected input times for motion estimation:")
for value in input_ds.time.values:
    print(" -", pd.to_datetime(value), "UTC")

Selected input times for motion estimation:
 - 2026-06-10 04:10:00 UTC
 - 2026-06-10 04:20:00 UTC
 - 2026-06-10 04:30:00 UTC
 - 2026-06-10 04:40:00 UTC
 - 2026-06-10 04:50:00 UTC
 - 2026-06-10 05:00:00 UTC
 - 2026-06-10 05:10:00 UTC
 - 2026-06-10 05:20:00 UTC
 - 2026-06-10 05:30:00 UTC
 - 2026-06-10 05:40:00 UTC
 - 2026-06-10 05:50:00 UTC
 - 2026-06-10 06:00:00 UTC
 - 2026-06-10 06:10:00 UTC
 - 2026-06-10 06:20:00 UTC
 - 2026-06-10 06:30:00 UTC
 - 2026-06-10 06:40:00 UTC
 - 2026-06-10 06:50:00 UTC
 - 2026-06-10 07:00:00 UTC
 - 2026-06-10 07:10:00 UTC
 - 2026-06-10 07:20:00 UTC
 - 2026-06-10 07:30:00 UTC
 - 2026-06-10 07:40:00 UTC
 - 2026-06-10 07:50:00 UTC
 - 2026-06-10 08:00:00 UTC
 - 2026-06-10 08:10:00 UTC
 - 2026-06-10 08:20:00 UTC
 - 2026-06-10 08:30:00 UTC
 - 2026-06-10 08:40:00 UTC
 - 2026-06-10 08:50:00 UTC
 - 2026-06-10 09:00:00 UTC
 - 2026-06-10 09:10:00 UTC
 - 2026-06-10 09:20:00 UTC
 - 2026-06-10 09:30:00 UTC
 - 2026-06-10 09:40:00 UTC
 - 2026-06-10 09:50:00 UTC


In [17]:
precip = wh.precip_array_from_input(input_ds, settings)

print("Input array shape (time, lat, lon):", precip.shape)
print("Min rainfall rate:", float(np.nanmin(precip)), "mm h-1")
print("Max rainfall rate:", float(np.nanmax(precip)), "mm h-1")
print("Wet pixels:", int(np.sum(precip >= settings["rain_threshold_mm_h"])))

Input array shape (time, lat, lon): (35, 700, 600)
Min rainfall rate: 0.0 mm h-1
Max rainfall rate: 43.400001525878906 mm h-1
Wet pixels: 1482583


## Step 9: Run the pySTEPS nowcast

The helper below first estimates the motion field with optical flow
(Lucas–Kanade) and then runs the STEPS stochastic ensemble nowcast. If
STEPS fails for any reason — for example because the rainfall input is
too dry — the helper falls back to deterministic extrapolation so the
rest of the notebook can still run.

**Run time.** With `n_ensemble_members = 8` and `n_lead_times = 6`, this
takes roughly half a minute on a typical laptop.

In [ ]:
forecast_array, velocity, nowcast_method, precip_log, pysteps_metadata = (
    wh.run_pysteps_nowcast(precip, settings)
)
print("Nowcast method:        ", nowcast_method)
print("Forecast array shape:  ", forecast_array.shape, "(member, lead_time, lat, lon)")

C:\CREWS_Lao_PDR_Cambodia\.pixi\envs\default\Lib\site-packages\pysteps\io\interface.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import iter_entry_points


Pysteps configuration file found at: C:\Users\Aerts\pysteps\pystepsrc



C:\CREWS_Lao_PDR_Cambodia\.pixi\envs\default\Lib\importlib\__init__.py:169: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  _bootstrap._exec(spec, module)


## Step 10: Visualise the estimated motion vectors

The arrows below show the estimated movement of rainfall features. With
10-minute ERR input, one timestep is 10 minutes. **Use these plots as
a diagnostic.** If the arrows look unrealistic (e.g. very short, very
random, or pointing inland during a known easterly monsoon flow), the
nowcast should be interpreted with caution.

In [ ]:
wh.plot_motion_vectors_bokeh(
    input_ds, velocity, settings,
    skip=20, arrow_scale=4.0,
)

### Motion vectors zoomed around each city

In [ ]:
wh.plot_motion_vectors_city_zooms_bokeh(
    input_ds, velocity, settings,
    skip=8, arrow_scale=2.0,
)

## Step 11: Convert the nowcast to xarray and save to NetCDF

In [ ]:
nowcast_ds = wh.nowcast_to_xarray(forecast_array, input_ds, settings, nowcast_method)
nowcast_ds

In [ ]:
nowcast_file = wh.save_nowcast_netcdf(
    nowcast_ds, folders["nowcast"], input_ds, nowcast_method,
)

## Step 12: Visualise the nowcast spatially

We plot the **ensemble-mean** forecast for the first lead time and the
**maximum** forecast rainfall rate over all lead times. The latter is a
useful single-panel summary for warning discussions.

A shared colour scale (`nowcast_vmax`) is computed across every panel so
the colours are directly comparable.

In [ ]:
forecast_mean = nowcast_ds["precipitation_rate_nowcast"].mean("member")
max_forecast  = forecast_mean.max("time")

nowcast_vmax = wh.compute_vmax(
    [forecast_mean.isel(time=i) for i in range(forecast_mean.sizes["time"])]
    + [max_forecast],
    percentile=settings["plot_vmax_percentile"],
    override=settings.get("plot_intensity_vmax"),
)
print(
    f"Shared nowcast intensity vmax: {nowcast_vmax:.2f} mm/h"
    if nowcast_vmax is not None
    else "Shared nowcast intensity vmax: no positive forecast pixels found."
)

### Ensemble-mean nowcast — lead time +10 min

In [ ]:
plot_field = forecast_mean.isel(time=0)

wh.plot_field_bokeh(
    plot_field,
    settings,
    title=(
        f"NOAA ERR nowcast (ensemble mean) | lead time {int(plot_field.lead_time.values)} x 10 min | "
        f"valid {pd.to_datetime(plot_field.time.values)} UTC"
    ),
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
)

### Maximum forecast rainfall rate over all lead times

In [ ]:
wh.plot_field_bokeh(
    max_forecast,
    settings,
    title="NOAA ERR nowcast — maximum rainfall rate over all lead times",
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
)

### Maximum nowcast rainfall rate — city zooms

In [ ]:
wh.plot_city_zooms_bokeh(
    max_forecast,
    settings,
    title_prefix="NOAA ERR nowcast — max over lead times",
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
)

### All forecast lead times on a shared colour scale

Each panel is one lead time (10 minutes apart). The colour scale is shared
across all panels so the colour of a pixel can be compared directly.

In [ ]:
wh.plot_nowcast_lead_times_grid_bokeh(
    forecast_mean, settings,
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
    ncols=3,
)

### Lead-time grid — city zooms

One grid per city, every lead time inside a small zoom box.

In [ ]:
wh.plot_nowcast_lead_times_grid_city_zooms_bokeh(
    forecast_mean, settings,
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
    ncols=3,
)

### Nowcast animation

A slider + play-button animation across the six lead times. Drag the
slider to step through manually; click ▶ to auto-advance.

In [ ]:
wh.plot_nowcast_animation_bokeh(
    forecast_mean, settings,
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
    interval_ms=800,
)

In [ ]:
wh.plot_nowcast_animation_city_zooms_bokeh(
    forecast_mean, settings,
    vmax=nowcast_vmax,
    cbar_label="mm h-1",
    interval_ms=800,
)

## Step 13: Combined observed + nowcast time series at each city

The plot below combines the **observed** ERR rainfall (in **blue**) and
the **pySTEPS nowcast** (in **red**) at each configured city, on a single
time axis. The dashed black vertical line marks the **nowcast start** —
the time of the last observation that pySTEPS used to estimate motion.
Everything to the left of the line is observed; everything to the right
is forecast.

Because the nowcast is an ensemble, every individual member is drawn as
a thin transparent red line and the ensemble mean is drawn on top as a
thick red line. The spread between the members is a direct, visual
indication of nowcast uncertainty.

Things to look for during the workshop:

1. Does the nowcast continue smoothly from the most recent observations,
   or does it jump?
2. How quickly does the ensemble spread (the band of thin red lines) widen
   with lead time?
3. Does the nowcast forecast *more* or *less* rainfall than the latest
   observations suggest?

> Click any legend entry to hide or show that series. Hover anywhere to
> read the exact values.

In [ ]:
wh.plot_obs_and_nowcast_timeseries_bokeh(
    obs_ds=err_ds,
    nowcast_ds=nowcast_ds,
    settings=settings,
    show_ensemble_members=True,
)

## End of workshop notebook

You now have:

- A processed NOAA ERR NetCDF for the workshop AOI in `./err_data/processed_netcdf/`.
- A saved pySTEPS ensemble nowcast NetCDF in `./err_data/nowcast/`.
- A complete set of interactive Bokeh plots illustrating both the raw
  observations and the pySTEPS nowcast at Phnom Penh and Vientiane.

### Suggested extensions for participants

- **Change the event window.** Try a known heavy-rain day; the same
  workflow runs unchanged.
- **Change the AOI.** Make the bounding box smaller to focus on a single
  catchment, or larger to see the upstream weather context.
- **Tune the nowcast.** Increase `n_input_files` for more stable motion
  estimates, increase `n_ensemble_members` for a smoother spread
  estimate.
- **Compare against rain gauges.** If DOM Cambodia or DMH Lao PDR AWS
  gauge data are available for the event, extract the nearest grid cell
  and overlay the gauge time series on the city plots above.